# 02. Generating Images with Azure OpenAI GPT-Image

This notebook demonstrates **text-to-image generation** using an Azure OpenAI image model
(the `gpt-image` family) via the standard OpenAI Python SDK pointed at an Azure OpenAI endpoint.
It generates a single training-course illustration from a text prompt and saves it to disk. This
belongs to the *Image Generation* part of the Image Generation/Content Safety section — the
starting point before editing (03, 04).

**Difficulty:** Beginner — this is the simplest possible image generation call: one prompt in,
one image out.

## Prerequisites

**Python packages** (already in the repo's root `requirements.txt` — no extra install needed):
- `openai` — the standard OpenAI Python SDK, reused against an Azure OpenAI endpoint
- `python-dotenv` — loads `.env`
- `base64`, `pathlib` — Python standard library

**Azure resources**
- An **Azure OpenAI** resource (or an Azure AI Foundry project's OpenAI-compatible endpoint)
  with an **image model deployment**, e.g. `gpt-image-1` / `gpt-image-2` or a `dall-e-3`
  deployment.

**Environment variables** — add these to the root `.env`:
```
AZURE_OPENAI_IMAGE_ENDPOINT=https://<your-resource>.services.ai.azure.com/openai/v1
AZURE_OPENAI_IMAGE_API_KEY=<your-api-key>
AZURE_OPENAI_IMAGE_DEPLOYMENT=gpt-image-2
```
These are new variable names (not currently in this repo's `.env`) because this script talks to
Azure OpenAI's image-generation endpoint directly with an API key, rather than through a Foundry
project + `DefaultAzureCredential` like notebook 01 does.

## What You'll Learn

- How to point the standard `openai` Python SDK at an **Azure OpenAI** endpoint (`base_url` +
  `api_key`) instead of the default OpenAI API
- The `images.generate()` call: `prompt`, `size`, `quality`, `n`, `output_format`
- Why the response comes back as base64 (`b64_json`) rather than a URL, and how to decode and
  save it to disk
- Key parameter differences between the `gpt-image` model family and DALL-E 3

### Imports and setup

- `base64` decodes the image data returned by the API.
- `Path` (from `pathlib`) is used for a clean, cross-platform way to write the output file.
- `OpenAI` is the standard SDK client class — reused here against an Azure endpoint.
- `load_dotenv()` loads the root `.env` so the config below can be read via `os.getenv`.

💡 **Exam tip:** AI-102 expects you to recognize that Azure OpenAI exposes an
**OpenAI-API-compatible** surface — the same `openai` SDK works against both OpenAI's own API
and an Azure OpenAI resource; only the `base_url`/auth and the model/deployment name change.

🔄 **Alternatives:** Instead of a raw API key, you could authenticate with Entra ID by supplying
an `azure_ad_token_provider` (via `azure-identity`) to `AzureOpenAI` — the same
credential-based pattern used for the Foundry project client in notebook 01.

In [ ]:
import os
import base64
from pathlib import Path

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

### Building the client

- `endpoint` is the Azure OpenAI **v1** API base URL — the unified, OpenAI-compatible surface
  (`/openai/v1`) rather than the older Azure-specific `/openai/deployments/<name>/...` URL shape.
- `api_key` is read from the environment instead of being hardcoded (the original script left it
  as an empty string placeholder).
- `IMAGE_DEPLOYMENT_NAME` names the specific image-model deployment to call — this is passed as
  the `model` parameter, standing in for what a plain OpenAI call would call the "model name".

💡 **Exam tip:** In Azure OpenAI, the `model` parameter in SDK calls refers to your
**deployment name**, which you choose when deploying a model in the Azure AI Foundry / Azure
OpenAI Studio — it does not have to match the underlying model's own name, though it commonly
does for clarity.

🔄 **Alternatives:** You could use `AzureOpenAI` (a subclass in the `openai` SDK purpose-built
for Azure, taking `azure_endpoint`, `api_version`, `azure_deployment`) instead of the generic
`OpenAI` class with a manually-constructed `base_url` — both work; the v1 API surface used here
is the newer, simpler option.

In [ ]:
endpoint = os.getenv("AZURE_OPENAI_IMAGE_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_IMAGE_API_KEY")
IMAGE_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_IMAGE_DEPLOYMENT", "gpt-image-2")

client = OpenAI(
    base_url=endpoint,
    api_key=api_key
)

### Generating the image

- `prompt` is the natural-language description of the desired image.
- `n=1` requests a single image (the API supports generating multiple variations in one call for
  some models).
- `size="1024x1024"` — the requested pixel dimensions; supported sizes depend on the model.
- `quality="medium"` — the `gpt-image` family uses `low` / `medium` / `high` (unlike DALL-E 3,
  which uses `standard` / `hd`).
- `output_format="png"` — requests PNG-encoded output (some models also support `jpeg`/`webp`).

💡 **Exam tip:** Know that image generation requests (and the resulting images) are subject to
Azure OpenAI's built-in **content filtering** just like text completions — a prompt describing
disallowed content can be rejected before an image is ever generated.

🔄 **Alternatives:** Swap the deployment name/model for a `dall-e-3` deployment to compare
output style and available parameters (DALL-E 3 also supports a `style` parameter — `vivid` or
`natural` — which `gpt-image` models do not).

In [ ]:
response = client.images.generate(
    model=IMAGE_DEPLOYMENT_NAME,
    prompt=(
        "Create a professional training image for an online course. "
        "Show a modern AI application dashboard with charts, documents, "
        "and an AI assistant helping business users. "
        "Use a clean corporate style suitable for a Microsoft Azure AI course."
    ),
    n=1,
    size="1024x1024",
    quality="medium",
    output_format="png"
)

### Decoding and saving the image

- `response.data[0].b64_json` — the first (and only) generated image, returned as a base64
  string rather than a hosted URL.
- `base64.b64decode(...)` turns that string back into raw PNG bytes.
- `Path("generated_image.png").write_bytes(...)` writes those bytes straight to disk in the
  current working directory — when you run this notebook, that means alongside this `.ipynb`
  file, overwriting the existing `generated_image.png` sample output already in this folder.

In [ ]:
image_base64 = response.data[0].b64_json
image_bytes = base64.b64decode(image_base64)

output_path = Path("generated_image.png")
output_path.write_bytes(image_bytes)

print(f"Image saved to: {output_path}")

## Summary

This notebook called Azure OpenAI's `images.generate()` endpoint with a text prompt, requesting a
single 1024x1024 PNG image at medium quality. The response returned the image as base64-encoded
JSON data (`b64_json`), which was decoded and written to `generated_image.png` in the working
directory. This is the foundation for the editing workflows in notebooks 03 and 04, which start
from an *existing* image instead of generating one from scratch.

## Try It Yourself

1. **Easy:** Change `quality` to `"low"` or `"high"` and compare output detail and (if you check
   your Azure billing) cost.
2. **Intermediate:** Request `n=2` or `n=3` and save each returned image with a distinct
   filename to compare variations from the same prompt.
3. **Advanced:** Rewrite the prompt to deliberately include something borderline (e.g. a
   recognizable brand logo or a public figure's likeness) and observe whether the request is
   rejected by content filtering — then explain why, referencing what you'll learn about content
   safety categories in notebooks 05-07.